# 🎬 Multi-Modal Video Analyzer - Line-by-Line Version

**Purpose:** Educational version with all code inline (no functions) for easy understanding.

**What This Demonstrates:**
- Multi-modal AI (combining vision + language)
- Computer vision with EasyOCR
- NLP with GPT-4
- Data fusion across modalities

**Note:** This version has NO function definitions after data classes. All code is written step-by-step so you can see exactly what each line does.

---

## 1️⃣ Install Required Packages

In [ ]:
# Install all required packages
!pip install opencv-python easyocr openai python-dotenv pillow numpy tqdm matplotlib ipywidgets

## 2️⃣ Import Libraries

In [ ]:
# Standard library imports
import os                      # Operating system operations (file paths, environment variables)
import json                    # JSON reading and writing
import re                      # Regular expressions for pattern matching
from pathlib import Path       # Object-oriented file paths
from typing import List, Dict, Any, Tuple  # Type hints for clarity
from collections import Counter            # Count occurrences of items
from dataclasses import dataclass, field   # Data classes for structured data
from datetime import datetime              # Timestamps for reports

# Third-party imports
import cv2                     # OpenCV - computer vision library
import numpy as np             # NumPy - numerical operations on arrays
import easyocr                 # EasyOCR - deep learning OCR
from openai import OpenAI      # OpenAI - GPT-4 API
from dotenv import load_dotenv # Load environment variables from .env
from tqdm.notebook import tqdm # Progress bars for Jupyter
import matplotlib.pyplot as plt # Plotting library for visualizations

# Display settings for Jupyter
from IPython.display import display, Markdown, HTML

print("✅ All libraries imported successfully!")

## 3️⃣ Configuration & Setup

In [ ]:
# Load environment variables from .env file
load_dotenv()

# Get OpenAI API key from environment
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Check if API key exists
if not OPENAI_API_KEY:
    print("⚠️  WARNING: OPENAI_API_KEY not found in .env file")
    print("Create a .env file with: OPENAI_API_KEY=your_key_here")
else:
    print("✅ OpenAI API key loaded")

# Initialize OpenAI client for GPT-4 API calls
client = OpenAI(api_key=OPENAI_API_KEY)

# Set up directory structure using Path objects
DATA_DIR = Path("./data")                    # Main data folder
VIDEOS_DIR = DATA_DIR / "videos"             # Input videos go here
FRAMES_DIR = DATA_DIR / "output" / "frames"  # Extracted frames saved here
REPORTS_DIR = DATA_DIR / "output" / "reports" # Final reports saved here

# Create directories if they don't exist
VIDEOS_DIR.mkdir(parents=True, exist_ok=True)   # parents=True creates parent dirs too
FRAMES_DIR.mkdir(parents=True, exist_ok=True)   # exist_ok=True doesn't error if exists
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n📁 Directory structure:")
print(f"  Videos: {VIDEOS_DIR}")
print(f"  Frames: {FRAMES_DIR}")
print(f"  Reports: {REPORTS_DIR}")

## 4️⃣ Data Classes (Structured Data Containers)

These classes define the structure of our data. Think of them like forms with specific fields.

In [ ]:
@dataclass
class OCRResult:
    """Single OCR detection result - one piece of detected text"""
    text: str                    # The actual text found
    confidence: float            # How confident the model is (0.0 to 1.0)
    bbox: List[List[int]]       # Bounding box coordinates (where the text is)
    frame_number: int            # Which video frame this came from
    
    def to_dict(self) -> Dict[str, Any]:
        """Convert to dictionary for JSON serialization"""
        return {
            'text': self.text,
            'confidence': float(self.confidence),  # Convert numpy types to Python types
            'bbox': [[int(x) for x in point] for point in self.bbox],
            'frame_number': int(self.frame_number)
        }


@dataclass
class VisualSummary:
    """Summary of all visual analysis (OCR results from all frames)"""
    total_frames: int                    # How many frames we analyzed
    frames_with_text: int               # How many had readable text
    unique_text_blocks: int             # Total text detections
    extracted_text: List[OCRResult]     # All OCR results
    key_frames: List[int]               # Frames with lots of text
    detected_terms: List[str]           # Important terms found (capitalized, acronyms)
    numbers_found: List[Dict[str, Any]] # Numbers, percentages, prices
    
    def to_dict(self) -> Dict[str, Any]:
        """Convert to dictionary for JSON serialization"""
        return {
            'total_frames': int(self.total_frames),
            'frames_with_text': int(self.frames_with_text),
            'unique_text_blocks': int(self.unique_text_blocks),
            'extracted_text': [r.to_dict() for r in self.extracted_text],
            'key_frames': [int(f) for f in self.key_frames],
            'detected_terms': self.detected_terms,
            'numbers_found': [
                {
                    'value': n['value'],
                    'numeric': float(n['numeric']),
                    'is_percentage': bool(n['is_percentage']),
                    'is_price': bool(n['is_price']),
                    'context': n['context'],
                    'frame': int(n['frame']),
                    'confidence': float(n['confidence'])
                }
                for n in self.numbers_found
            ]
        }


@dataclass
class TextSummary:
    """Summary of text analysis (NLP results from GPT-4)"""
    summary: str                        # Brief overview of content
    tone: str                           # positive, negative, or neutral
    claims: List[Dict[str, Any]]       # Extracted claims with types
    entities: List[str]                # Mentioned topics, tools, concepts
    key_points: List[str]              # Main takeaways
    
    def to_dict(self) -> Dict[str, Any]:
        """Convert to dictionary for JSON serialization"""
        return {
            'summary': self.summary,
            'tone': self.tone,
            'claims': self.claims,
            'entities': self.entities,
            'key_points': self.key_points
        }


@dataclass
class MultiModalReport:
    """Final fused report combining visual and text analysis"""
    video_name: str                     # Name of analyzed video
    timestamp: str                      # When analysis was done
    quality_score: float                # Overall quality (0-100)
    visual_summary: Dict[str, Any]      # Visual analysis results
    text_summary: Dict[str, Any]        # Text analysis results
    all_entities: List[str]             # Combined entities from both
    insights: List[str]                 # Generated insights
    
    def to_dict(self) -> Dict[str, Any]:
        """Convert to dictionary for JSON serialization"""
        return {
            'video_name': self.video_name,
            'timestamp': self.timestamp,
            'quality_score': self.quality_score,
            'visual_summary': self.visual_summary,
            'text_summary': self.text_summary,
            'all_entities': self.all_entities,
            'insights': self.insights
        }

print("✅ Data classes defined")

---

# 🚀 START OF ANALYSIS PIPELINE

From here on, everything is **inline code** - no function definitions!

---

## STEP 1: Select Video to Analyze

In [ ]:
# Find all video files in the videos directory
# glob() returns all files matching the pattern
available_videos = list(VIDEOS_DIR.glob("*.mp4")) + list(VIDEOS_DIR.glob("*.mkv"))

# Check if we found any videos
if available_videos:
    print("📹 Available videos:")
    
    # Loop through and display each video with size
    for i, vid in enumerate(available_videos):
        # Get file size in megabytes
        size_mb = vid.stat().st_size / (1024 * 1024)  # Convert bytes to MB
        print(f"  {i}: {vid.name} ({size_mb:.1f} MB)")
    
    # SELECT VIDEO HERE - change the index to select different video
    video_index = 0  # 👈 CHANGE THIS NUMBER to select different video
    video_path = available_videos[video_index]
    
    print(f"\n✅ Selected: {video_path.name}")
else:
    print("⚠️  No videos found!")
    print(f"Place video files (.mp4 or .mkv) in: {VIDEOS_DIR}")
    video_path = None

## STEP 2: Extract Frames from Video

We'll sample frames at 0.5 FPS (1 frame every 2 seconds) to reduce processing time.

In [ ]:
# Only proceed if we have a video selected
if video_path:
    # CONFIGURATION
    target_fps = 0.5  # Extract 1 frame every 2 seconds (0.5 FPS)
    
    # Clear any existing frames from previous runs
    print("🧹 Clearing old frames...")
    for old_frame in FRAMES_DIR.glob("*.jpg"):
        old_frame.unlink()  # Delete the file
    
    print(f"🎬 Opening video: {video_path.name}")
    
    # Open video file with OpenCV
    # VideoCapture creates an object that can read the video
    cap = cv2.VideoCapture(str(video_path))  # Convert Path to string
    
    # Check if video opened successfully
    if not cap.isOpened():
        print(f"❌ Could not open video: {video_path}")
    else:
        # Get video properties using CAP_PROP constants
        video_fps = cap.get(cv2.CAP_PROP_FPS)           # Frames per second
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))  # Total frames
        duration = total_frames / video_fps             # Duration in seconds
        
        print(f"📊 Video info:")
        print(f"  FPS: {video_fps:.2f}")                # Show 2 decimal places
        print(f"  Total frames: {total_frames:,}")      # Add commas for thousands
        print(f"  Duration: {duration:.1f} seconds ({duration/60:.1f} minutes)")
        print(f"  Extracting at: {target_fps} FPS (1 frame every {1/target_fps:.0f} seconds)")
        
        # Calculate how many frames to skip between saves
        # If video is 30 FPS and we want 0.5 FPS: 30 / 0.5 = 60
        # So we save every 60th frame
        frame_interval = int(video_fps / target_fps)
        
        # Estimate how many frames we'll extract
        expected_frames = total_frames // frame_interval
        print(f"  Expected frames to extract: ~{expected_frames}")
        
        # Initialize counters
        frame_count = 0   # Tracks which frame we're on
        saved_count = 0   # How many frames we've saved
        
        # Create progress bar for visual feedback
        # tqdm shows a progress bar in Jupyter
        with tqdm(total=expected_frames, desc="Extracting frames") as pbar:
            # Infinite loop - we'll break when video ends
            while True:
                # Read next frame from video
                # ret = boolean (True if frame read successfully)
                # frame = numpy array containing the image
                ret, frame = cap.read()
                
                # If ret is False, we've reached the end of video
                if not ret:
                    break
                
                # Check if this frame should be saved
                # Modulo (%) gives remainder of division
                # If frame_count is divisible by frame_interval, remainder is 0
                if frame_count % frame_interval == 0:
                    # Create filename with zero-padded number (0001, 0002, etc.)
                    frame_filename = FRAMES_DIR / f"frame_{saved_count:04d}.jpg"
                    
                    # Write frame as JPEG image
                    # frame is a numpy array, imwrite saves it
                    cv2.imwrite(str(frame_filename), frame)
                    
                    saved_count += 1      # Increment saved counter
                    pbar.update(1)        # Update progress bar
                
                frame_count += 1          # Always increment frame counter
        
        # Release the video file (close it properly)
        cap.release()
        
        print(f"\n✅ Extracted {saved_count} frames to {FRAMES_DIR}")
        
        # Store for later use
        num_frames_extracted = saved_count

else:
    print("⚠️  No video selected - skipping frame extraction")

## STEP 3: Display Sample Frames

Let's visualize some of the extracted frames to see what we're working with.

In [ ]:
# Get all extracted frame files
frame_files = sorted(list(FRAMES_DIR.glob("*.jpg")))

if frame_files:
    # How many samples to show
    num_samples = 4
    
    # Select evenly spaced samples across the video
    if len(frame_files) <= num_samples:
        # If we have fewer frames than samples, show all
        samples = frame_files
    else:
        # Calculate step size for even spacing
        step = len(frame_files) // num_samples
        # Select frames: [0, step, 2*step, 3*step]
        samples = [frame_files[i * step] for i in range(num_samples)]
    
    # Create matplotlib figure with subplots
    # 1 row, len(samples) columns, 15 inches wide, 4 inches tall
    fig, axes = plt.subplots(1, len(samples), figsize=(15, 4))
    
    # If only 1 sample, axes is not a list, so wrap it
    if len(samples) == 1:
        axes = [axes]
    
    # Plot each sample frame
    for ax, frame_path in zip(axes, samples):
        # Read image with OpenCV
        frame = cv2.imread(str(frame_path))
        
        # OpenCV reads images as BGR, matplotlib expects RGB
        # So we need to convert: Blue, Green, Red → Red, Green, Blue
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Display image in subplot
        ax.imshow(frame_rgb)
        
        # Set title to frame filename (without extension)
        ax.set_title(frame_path.stem)  # stem = filename without .jpg
        
        # Turn off axis labels (cleaner look)
        ax.axis('off')
    
    # Adjust spacing between subplots
    plt.tight_layout()
    
    # Display the figure
    plt.show()
    
    print(f"Showing {len(samples)} sample frames from {len(frame_files)} total frames")
else:
    print("No frames found to display")

## STEP 4: Visual Analysis with OCR

Now we'll use EasyOCR to extract text from all the frames. This uses deep learning!

In [ ]:
# CONFIGURATION
confidence_threshold = 0.3  # Only keep detections with 30%+ confidence

# Initialize EasyOCR reader
print("🔄 Initializing EasyOCR reader...")
print("(Downloads neural network models on first run - may take 1-2 minutes)")

# Create reader for English language
# gpu=False uses CPU (set to True if you have NVIDIA GPU)
# verbose=False suppresses detailed output
reader = easyocr.Reader(['en'], gpu=False, verbose=False)

print("✅ OCR reader initialized\n")

# Get all frame files (JPG and PNG)
frame_files = sorted(list(FRAMES_DIR.glob("*.jpg")) + list(FRAMES_DIR.glob("*.png")))

if not frame_files:
    print("⚠️  No frames found to analyze")
else:
    print(f"🔍 Analyzing {len(frame_files)} frames with OCR...")
    print(f"   Confidence threshold: {confidence_threshold}\n")
    
    # Lists to store results
    all_ocr_results = []    # All OCR detections
    frames_with_text = 0    # Count frames that had text
    
    # Process each frame with progress bar
    for frame_number, frame_file in enumerate(tqdm(frame_files, desc="OCR Analysis")):
        # Read image with OpenCV
        image = cv2.imread(str(frame_file))
        
        # Skip if image couldn't be read
        if image is None:
            continue
        
        # ===== PREPROCESSING =====
        # Convert color image to grayscale
        # Grayscale is simpler for OCR (1 channel instead of 3)
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        
        # Apply sharpening filter to enhance edges
        # This is a convolution kernel (3x3 matrix)
        sharpening_kernel = np.array([[-1, -1, -1],
                                       [-1,  9, -1],
                                       [-1, -1, -1]])
        
        # filter2D applies the kernel to the image
        # -1 means output has same depth as input
        processed = cv2.filter2D(gray, -1, sharpening_kernel)
        
        # ===== OCR DETECTION =====
        # This is where the deep learning happens!
        # readtext() returns a list of detections
        # Each detection is: (bounding_box, text, confidence)
        detections = reader.readtext(processed)
        
        # Process each detection
        frame_results = []  # Store results for this frame
        
        for bbox, text, confidence in detections:
            # Only keep if confidence is above threshold
            if confidence >= confidence_threshold:
                # Create OCRResult object
                ocr_result = OCRResult(
                    text=text.strip(),            # Remove whitespace
                    confidence=float(confidence), # Ensure it's a Python float
                    bbox=bbox,                    # Bounding box coordinates
                    frame_number=frame_number     # Which frame this came from
                )
                frame_results.append(ocr_result)
        
        # If we found any text in this frame, count it
        if frame_results:
            frames_with_text += 1
            all_ocr_results.extend(frame_results)  # Add to overall list
    
    print(f"\n✅ OCR complete!")
    print(f"   Frames with text: {frames_with_text}/{len(frame_files)}")
    print(f"   Total text blocks detected: {len(all_ocr_results)}")

## STEP 5: Extract Key Terms from OCR Results

Find important terms like capitalized words and acronyms.

In [ ]:
# Create a set to store unique terms (sets automatically remove duplicates)
detected_terms = set()

# Regular expression pattern to find capitalized words and acronyms
# \b = word boundary
# [A-Z][A-Z]+ = 2 or more capital letters (acronyms like API, ML)
# [A-Z][a-z]+ = Capital followed by lowercase (proper nouns like Python)
term_pattern = re.compile(r'\b[A-Z][A-Z]+\b|\b[A-Z][a-z]+\b')

# Words to exclude (common words that aren't important)
excluded_words = {
    'THE', 'AND', 'FOR', 'WITH', 'FROM', 'THIS', 'THAT',
    'ARE', 'WAS', 'NOT', 'BUT', 'CAN', 'YOU', 'ALL', 'NEW',
    'NOW', 'OUT', 'GET', 'GOT', 'HAS', 'HAD', 'HOW', 'WHY',
    'WHEN', 'WHERE', 'WHAT', 'WHO'
}

# Go through each OCR result
for result in all_ocr_results:
    # Find all matches of the pattern in the text
    matches = term_pattern.findall(result.text)
    
    # Check each match
    for match in matches:
        # Only include if:
        # 1. Not in excluded words
        # 2. Length is at least 2 characters
        if match not in excluded_words and len(match) >= 2:
            detected_terms.add(match)  # Add to set

# Convert set to sorted list
detected_terms = sorted(list(detected_terms))

print(f"🏷️  Found {len(detected_terms)} key terms")
print(f"   Sample terms: {', '.join(detected_terms[:10])}")

## STEP 6: Extract Numbers from OCR Results

Find structured numerical data like percentages, prices, and numbers.

In [ ]:
# List to store all found numbers
numbers_found = []

# Regular expression patterns for different number formats
number_patterns = [
    r'\$[\d,]+\.?\d*[kKmMbB]?',  # Prices: $95,000 or $1.2M
    r'-?\d+\.?\d*%',              # Percentages: 25% or -3.5%
    r'-?\d+\.?\d*[kKmMbB]',       # Abbreviated: 95k, 1.2M, 5B
    r'-?\d{1,3}(?:,\d{3})*(?:\.\d+)?',  # Comma-separated: 1,234.56
    r'-?\d+\.?\d*',               # Plain numbers: 123 or 45.67
]

# Go through each OCR result
for result in all_ocr_results:
    # Try each pattern
    for pattern in number_patterns:
        # Find all matches of this pattern
        matches = re.findall(pattern, result.text)
        
        # Process each match
        for match in matches:
            # Check what type of number this is
            is_percentage = '%' in match
            is_price = '$' in match
            
            # Clean the string for numeric conversion
            # Remove: $, commas, %
            value_clean = match.replace('$', '').replace(',', '').replace('%', '')
            
            # Handle k, M, B abbreviations (thousands, millions, billions)
            multiplier = 1  # Default multiplier
            
            # Check if last character is k, M, or B
            if value_clean and value_clean[-1].upper() in ['K', 'M', 'B']:
                # Dictionary of multipliers
                multipliers = {'K': 1000, 'M': 1000000, 'B': 1000000000}
                # Get multiplier for this letter
                multiplier = multipliers.get(value_clean[-1].upper(), 1)
                # Remove the letter from the string
                value_clean = value_clean[:-1]
            
            # Try to convert to number
            try:
                # Convert to float and apply multiplier
                numeric_value = float(value_clean) * multiplier
                
                # Create number record
                number_record = {
                    'value': match,                  # Original string
                    'numeric': numeric_value,        # Numeric value
                    'is_percentage': is_percentage,  # Is it a percentage?
                    'is_price': is_price,           # Is it a price?
                    'context': result.text,         # Full text it came from
                    'frame': result.frame_number,   # Which frame
                    'confidence': result.confidence # OCR confidence
                }
                
                numbers_found.append(number_record)
                
            except ValueError:
                # If conversion fails, skip this number
                continue

print(f"🔢 Found {len(numbers_found)} numbers")
if numbers_found:
    print(f"   Sample: {numbers_found[0]['value']} (numeric: {numbers_found[0]['numeric']})")

## STEP 7: Identify Key Frames

Find frames with above-average text content.

In [ ]:
# Count how many text detections per frame
# Counter creates a dict: {frame_number: count}
frame_text_count = Counter(r.frame_number for r in all_ocr_results)

if frame_text_count:
    # Calculate average text blocks per frame
    # sum() adds all counts, len() gives number of frames
    avg_text = sum(frame_text_count.values()) / len(frame_text_count)
    
    # Find frames with more than 1.5x average text
    # These are "key frames" with dense content
    key_frames = [
        frame for frame, count in frame_text_count.items()
        if count > avg_text * 1.5
    ]
    
    # Sort frame numbers
    key_frames = sorted(key_frames)
else:
    key_frames = []

print(f"🎯 Identified {len(key_frames)} key frames with dense text content")

## STEP 8: Create Visual Summary Object

Package all visual analysis results into a VisualSummary object.

In [ ]:
# Create VisualSummary object with all our results
visual_summary = VisualSummary(
    total_frames=len(frame_files),           # How many frames analyzed
    frames_with_text=frames_with_text,      # How many had text
    unique_text_blocks=len(all_ocr_results), # Total detections
    extracted_text=all_ocr_results,         # All OCR results
    key_frames=key_frames,                  # Frames with dense text
    detected_terms=detected_terms,          # Key terms found
    numbers_found=numbers_found             # Numbers found
)

print("✅ Visual summary created")
print(f"   Total frames: {visual_summary.total_frames}")
print(f"   Frames with text: {visual_summary.frames_with_text}")
print(f"   Key terms: {len(visual_summary.detected_terms)}")
print(f"   Numbers: {len(visual_summary.numbers_found)}")

## STEP 9: Display Visual Analysis Results

In [ ]:
# Display results using Markdown formatting
display(Markdown("## 👁️ Visual Analysis Results"))
display(Markdown(f"**Frames analyzed:** {visual_summary.total_frames}"))
display(Markdown(f"**Frames with text:** {visual_summary.frames_with_text}"))

# Show key terms (first 10)
if visual_summary.detected_terms:
    display(Markdown(f"**Key terms detected:** {', '.join(visual_summary.detected_terms[:10])}"))
    if len(visual_summary.detected_terms) > 10:
        display(Markdown(f"*...and {len(visual_summary.detected_terms) - 10} more*"))

# Show numbers found
display(Markdown(f"**Numbers found:** {len(visual_summary.numbers_found)}"))

# Show sample numbers
if visual_summary.numbers_found:
    display(Markdown("\n**Sample data points:**"))
    # Show first 5 numbers
    for num in visual_summary.numbers_found[:5]:
        # Truncate context to 50 characters
        context = num['context'][:50]
        display(Markdown(f"- {num['value']} (from: {context}...)"))

---

# TEXT ANALYSIS (NLP with GPT-4)

---

## STEP 10: Prepare Transcript

In production, you'd use Whisper AI to transcribe the audio. For this demo, we use a sample transcript.

In [ ]:
# Sample transcript (replace with actual transcript from Whisper AI)
sample_transcript = """
Welcome to this Python programming tutorial. Today we'll cover data structures,
functions, and best practices for writing clean code.

First, let's discuss lists and dictionaries. Lists are ordered collections that
can store multiple items. For example, numbers = [1, 2, 3, 4, 5]. Dictionaries
store key-value pairs, which is useful for organizing related data.

Functions should do one thing well. We use the 'def' keyword to define them.
Always include docstrings explaining what each function does - this makes your
code maintainable.

Error handling is crucial. Use try-except blocks to handle potential errors
gracefully. This prevents crashes when something unexpected happens.

For performance, use list comprehensions instead of traditional loops. They make
code faster and more readable.

In conclusion, writing clean Python code involves understanding data structures,
writing focused functions, handling errors properly, and optimizing where needed.
"""

print("📝 Using sample transcript")
print(f"   Length: {len(sample_transcript)} characters")
print(f"   Words: ~{len(sample_transcript.split())} words")

## STEP 11: Analyze Tone with GPT-4

In [ ]:
print("🤖 Running NLP Analysis...\n")
print("1️⃣ Analyzing tone...")

# Create prompt for GPT-4
# We're asking for ONLY one word: POSITIVE, NEGATIVE, or NEUTRAL
tone_prompt = f"""
Analyze the tone of this content.

Return ONLY one word: POSITIVE, NEGATIVE, or NEUTRAL

Content:
{sample_transcript}
"""

# Make API call to GPT-4
tone_response = client.chat.completions.create(
    model="gpt-4",                                  # Use GPT-4 model
    messages=[{"role": "user", "content": tone_prompt}],  # Our prompt
    temperature=0.3,                                # Low temp = more consistent
    max_tokens=10                                   # Only need 1 word
)

# Extract the response text
content_tone = tone_response.choices[0].message.content.strip()

print(f"   Result: {content_tone}\n")

## STEP 12: Generate Summary with GPT-4

In [ ]:
print("2️⃣ Generating summary...")

# Create prompt for summary
summary_prompt = f"""
Create a concise 2-3 sentence summary of this content.

Content:
{sample_transcript}
"""

# Make API call to GPT-4
summary_response = client.chat.completions.create(
    model="gpt-4",
    messages=[{"role": "user", "content": summary_prompt}],
    temperature=0.5,      # Slightly higher for more natural language
    max_tokens=150        # Enough for 2-3 sentences
)

# Extract summary
content_summary = summary_response.choices[0].message.content.strip()

print(f"   {content_summary}\n")

## STEP 13: Extract Claims with GPT-4

This is where prompt engineering shines - we ask for structured JSON output!

In [ ]:
print("3️⃣ Extracting claims...")

# Create prompt for claim extraction
# Notice how we specify the exact JSON structure we want
claims_prompt = f"""
Extract all specific claims from this content.

For each claim, provide:
- claim_text: The actual claim
- claim_type: "factual", "prediction", or "opinion"
- confidence: How verifiable (0-100)

Return as a JSON array.

Content:
{sample_transcript}
"""

# Make API call
claims_response = client.chat.completions.create(
    model="gpt-4",
    messages=[{"role": "user", "content": claims_prompt}],
    temperature=0.3,
    max_tokens=1000  # More tokens for multiple claims
)

# Parse JSON response
try:
    # Extract content
    claims_text = claims_response.choices[0].message.content
    # Parse as JSON
    extracted_claims = json.loads(claims_text)
except:
    # If parsing fails, use empty list
    extracted_claims = []

print(f"   Found {len(extracted_claims)} claims\n")

## STEP 14: Extract Entities with GPT-4

In [ ]:
print("4️⃣ Identifying entities...")

# Create prompt for entity extraction
entities_prompt = f"""
List all important entities mentioned in this content.
Include: topics, tools, technologies, concepts, names.

Return as a JSON array of strings.

Content:
{sample_transcript}
"""

# Make API call
entities_response = client.chat.completions.create(
    model="gpt-4",
    messages=[{"role": "user", "content": entities_prompt}],
    temperature=0.3,
    max_tokens=200
)

# Parse JSON
try:
    entities_text = entities_response.choices[0].message.content
    extracted_entities = json.loads(entities_text)
except:
    extracted_entities = []

print(f"   Found: {', '.join(extracted_entities[:5])}...\n")

## STEP 15: Extract Key Points with GPT-4

In [ ]:
print("5️⃣ Extracting key points...")

# Create prompt
keypoints_prompt = f"""
Extract 3-5 key points or main takeaways from this content.

Return as a JSON array of strings.

Content:
{sample_transcript}
"""

# Make API call
keypoints_response = client.chat.completions.create(
    model="gpt-4",
    messages=[{"role": "user", "content": keypoints_prompt}],
    temperature=0.3,
    max_tokens=300
)

# Parse JSON
try:
    keypoints_text = keypoints_response.choices[0].message.content
    extracted_keypoints = json.loads(keypoints_text)
except:
    extracted_keypoints = []

print(f"   {len(extracted_keypoints)} key points\n")

print("✅ NLP analysis complete!")

## STEP 16: Create Text Summary Object

In [ ]:
# Package all NLP results into TextSummary object
text_summary = TextSummary(
    summary=content_summary,
    tone=content_tone,
    claims=extracted_claims,
    entities=extracted_entities,
    key_points=extracted_keypoints
)

print("✅ Text summary created")
print(f"   Tone: {text_summary.tone}")
print(f"   Entities: {len(text_summary.entities)}")
print(f"   Claims: {len(text_summary.claims)}")
print(f"   Key points: {len(text_summary.key_points)}")

## STEP 17: Display Text Analysis Results

In [ ]:
display(Markdown("## 💬 Text Analysis Results"))
display(Markdown(f"**Tone:** {text_summary.tone}"))
display(Markdown(f"**Summary:** {text_summary.summary}"))
display(Markdown(f"**Entities found:** {', '.join(text_summary.entities)}"))
display(Markdown(f"**Claims extracted:** {len(text_summary.claims)}"))

# Show key points
if text_summary.key_points:
    display(Markdown("\n**Key Points:**"))
    for i, point in enumerate(text_summary.key_points, 1):
        display(Markdown(f"{i}. {point}"))

---

# MULTI-MODAL FUSION

Now we combine visual and text analysis!

---

## STEP 18: Calculate Quality Score

Multi-criteria scoring based on visual evidence, claim quality, and completeness.

In [ ]:
print("🔄 Fusing multi-modal data...\n")

# Initialize score
quality_score = 0

# ===== VISUAL EVIDENCE SCORE (max 30 points) =====
if visual_summary.total_frames > 0:
    # Calculate ratio of frames with text
    text_ratio = visual_summary.frames_with_text / visual_summary.total_frames
    # Convert to score out of 30
    visual_score = min(text_ratio * 100, 30)
    quality_score += visual_score

# ===== CLAIMS SCORE (max 40 points) =====
if text_summary.claims:
    # Count factual claims
    factual_claims = sum(1 for c in text_summary.claims if c.get('claim_type') == 'factual')
    total_claims = len(text_summary.claims)
    # Calculate ratio
    factual_ratio = factual_claims / total_claims
    # Convert to score out of 40
    claims_score = factual_ratio * 40
    quality_score += claims_score

# ===== COMPLETENESS SCORE (max 30 points) =====
# Check if we have all components
has_summary = len(text_summary.summary) > 0
has_entities = len(text_summary.entities) > 0
has_key_points = len(text_summary.key_points) > 0

# Count how many we have (each worth 1/3 of 30 points)
completeness = sum([has_summary, has_entities, has_key_points]) / 3
completeness_score = completeness * 30
quality_score += completeness_score

# Make sure score doesn't exceed 100
quality_score = min(quality_score, 100)

print(f"📊 Quality Score: {quality_score:.1f}/100")

## STEP 19: Combine Entities from Both Modalities

In [ ]:
# Combine visual terms and text entities
# Using set() automatically removes duplicates
all_entities = list(set(
    visual_summary.detected_terms + text_summary.entities
))

# Sort alphabetically
all_entities = sorted(all_entities)

print(f"🏷️  Total unique entities: {len(all_entities)}")

## STEP 20: Find Cross-Modal Correlations

In [ ]:
# Convert to sets for intersection operation
visual_terms_set = set(visual_summary.detected_terms)
text_entities_set = set(text_summary.entities)

# Find overlap (entities in BOTH sources)
# intersection() returns items that appear in both sets
overlap = visual_terms_set.intersection(text_entities_set)

print(f"🔗 Cross-modal overlap: {len(overlap)} entities confirmed in both visual and text")
if overlap:
    print(f"   Examples: {', '.join(list(overlap)[:5])}")

## STEP 21: Generate Insights

In [ ]:
# List to store automatically generated insights
insights = []

# ===== VISUAL INSIGHTS =====
if visual_summary.numbers_found:
    insights.append(
        f"Found {len(visual_summary.numbers_found)} data points in visual content"
    )

if visual_summary.key_frames:
    insights.append(
        f"Identified {len(visual_summary.key_frames)} key frames with dense content"
    )

# ===== TEXT INSIGHTS =====
if text_summary.claims:
    # Count by type
    factual = sum(1 for c in text_summary.claims if c.get('claim_type') == 'factual')
    predictions = sum(1 for c in text_summary.claims if c.get('claim_type') == 'prediction')
    opinions = sum(1 for c in text_summary.claims if c.get('claim_type') == 'opinion')
    
    insights.append(
        f"Extracted {len(text_summary.claims)} claims "
        f"({factual} factual, {predictions} predictions, {opinions} opinions)"
    )

# ===== CROSS-MODAL INSIGHTS =====
if overlap:
    insights.append(
        f"Found {len(overlap)} entities confirmed across both visual and text analysis"
    )

print(f"💡 Generated {len(insights)} insights\n")
for insight in insights:
    print(f"   • {insight}")

print("\n✅ Multi-modal fusion complete!")

## STEP 22: Create Final Multi-Modal Report

In [ ]:
# Get current timestamp
current_timestamp = datetime.now().isoformat()

# Create final report object
final_report = MultiModalReport(
    video_name=video_path.name if video_path else "Unknown",
    timestamp=current_timestamp,
    quality_score=quality_score,
    visual_summary=visual_summary.to_dict(),
    text_summary=text_summary.to_dict(),
    all_entities=all_entities,
    insights=insights
)

print("✅ Final multi-modal report created!")

## STEP 23: Display Complete Report

In [ ]:
# Display formatted report
display(Markdown(f"# 📋 Multi-Modal Analysis Report"))
display(Markdown(f"**Video:** {final_report.video_name}"))
display(Markdown(f"**Analysis Date:** {final_report.timestamp}"))
display(Markdown(f"---"))

# Quality Score with color coding
if final_report.quality_score >= 70:
    score_emoji = "🟢"
elif final_report.quality_score >= 50:
    score_emoji = "🟡"
else:
    score_emoji = "🔴"

display(Markdown(f"## 📊 Quality Score: {score_emoji} {final_report.quality_score:.1f}/100"))

# Summary
display(Markdown(f"## 📝 Summary"))
display(Markdown(final_report.text_summary['summary']))

# Content Tone
tone = final_report.text_summary['tone'].upper()
tone_emoji = "😊" if tone == "POSITIVE" else "😐" if tone == "NEUTRAL" else "😟"
display(Markdown(f"**Tone:** {tone_emoji} {tone}"))

# Entities
display(Markdown(f"## 🏷️ Detected Entities ({len(final_report.all_entities)})"))
# Show first 20
display(Markdown(", ".join(final_report.all_entities[:20])))
if len(final_report.all_entities) > 20:
    display(Markdown(f"*... and {len(final_report.all_entities) - 20} more*"))

# Key Points
if final_report.text_summary['key_points']:
    display(Markdown(f"## 🎯 Key Points"))
    for i, point in enumerate(final_report.text_summary['key_points'], 1):
        display(Markdown(f"{i}. {point}"))

# Claims
if final_report.text_summary['claims']:
    display(Markdown(f"## 📋 Extracted Claims ({len(final_report.text_summary['claims'])})"))
    
    # Group by type
    factual = [c for c in final_report.text_summary['claims'] if c.get('claim_type') == 'factual']
    predictions = [c for c in final_report.text_summary['claims'] if c.get('claim_type') == 'prediction']
    opinions = [c for c in final_report.text_summary['claims'] if c.get('claim_type') == 'opinion']
    
    if factual:
        display(Markdown(f"### Factual Claims ({len(factual)})"))
        for claim in factual[:5]:  # Show first 5
            conf = claim.get('confidence', 0)
            display(Markdown(f"- {claim['claim_text']} *(confidence: {conf}%)*"))
    
    if predictions:
        display(Markdown(f"### Predictions ({len(predictions)})"))
        for claim in predictions[:5]:
            conf = claim.get('confidence', 0)
            display(Markdown(f"- {claim['claim_text']} *(confidence: {conf}%)*"))

# Insights
if final_report.insights:
    display(Markdown(f"## 💡 Insights"))
    for insight in final_report.insights:
        display(Markdown(f"- {insight}"))

# Visual Stats
display(Markdown(f"## 🎬 Visual Analysis Statistics"))
display(Markdown(f"- Total frames analyzed: {final_report.visual_summary['total_frames']}"))
display(Markdown(f"- Frames with text: {final_report.visual_summary['frames_with_text']}"))
display(Markdown(f"- Text blocks extracted: {final_report.visual_summary['unique_text_blocks']}"))
display(Markdown(f"- Data points found: {len(final_report.visual_summary['numbers_found'])}"))

## STEP 24: Save Report to Files

In [ ]:
# Create timestamp for filenames (YYYYMMDD_HHMMSS format)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# ===== SAVE JSON =====
json_filename = f"report_{timestamp}.json"
json_path = REPORTS_DIR / json_filename

# Open file for writing
with open(json_path, 'w', encoding='utf-8') as f:
    # Convert report to dict and write as JSON
    # indent=2 makes it readable
    # ensure_ascii=False allows unicode characters
    json.dump(final_report.to_dict(), f, indent=2, ensure_ascii=False)

print(f"✅ JSON report saved: {json_path}")

# ===== SAVE MARKDOWN =====
md_filename = f"report_{timestamp}.md"
md_path = REPORTS_DIR / md_filename

# Build markdown content string
md_content = f"""# Multi-Modal Analysis Report

**Video:** {final_report.video_name}
**Date:** {final_report.timestamp}
**Quality Score:** {final_report.quality_score:.1f}/100

---

## Summary

{final_report.text_summary['summary']}

**Tone:** {final_report.text_summary['tone'].upper()}

## Entities Detected

{', '.join(final_report.all_entities)}

## Key Points

"""

# Add key points
for i, point in enumerate(final_report.text_summary['key_points'], 1):
    md_content += f"{i}. {point}\n"

md_content += "\n## Insights\n\n"

# Add insights
for insight in final_report.insights:
    md_content += f"- {insight}\n"

md_content += f"""
## Statistics

- Frames analyzed: {final_report.visual_summary['total_frames']}
- Frames with text: {final_report.visual_summary['frames_with_text']}
- Claims extracted: {len(final_report.text_summary['claims'])}
- Entities found: {len(final_report.all_entities)}
"""

# Write markdown file
with open(md_path, 'w', encoding='utf-8') as f:
    f.write(md_content)

print(f"✅ Markdown report saved: {md_path}")
print(f"\n✅ Reports saved to {REPORTS_DIR}")

---

# ✨ Analysis Complete!

## What We Did:

1. ✅ **Frame Extraction** - Sampled video frames using OpenCV
2. ✅ **Visual Analysis (OCR)** - Extracted text with EasyOCR deep learning
3. ✅ **Pattern Extraction** - Found key terms and numbers with regex
4. ✅ **Text Analysis (NLP)** - Analyzed content with GPT-4
5. ✅ **Multi-Modal Fusion** - Combined visual + text with correlation
6. ✅ **Quality Scoring** - Multi-criteria evaluation
7. ✅ **Report Generation** - Exported as JSON and Markdown

## Files Generated:

- **Frames:** `data/output/frames/` (extracted images)
- **Reports:** `data/output/reports/` (JSON + Markdown)

---

**Every line of code was visible and explained! 🎓**

This is how multi-modal AI works - step by step, no hidden functions!